# 1. Data Collection (Thu thập dữ liệu)

- **Chủ thể (Subject):** Dữ liệu chi trả công nợ và mua sắm công (Accounts Payable) của chính quyền Thành phố Boulder, bang Colorado, Hoa Kỳ. Dữ liệu phản ánh các khoản giải ngân thực tế cho các nhà cung ứng bên ngoài phục vụ vận hành đô thị và đầu tư công.
- **Nguồn (Source):** Cổng dữ liệu mở chính thức của thành phố: [City of Boulder Open Data - Accounts Payable Current](https://open-data.bouldercolorado.gov/datasets/Boulder::accounts-payable-current/about). Ngày phát hành: 04/02/2025; Ngày cập nhật dữ liệu khai thác: 28/10/2026.
- **Thời gian & Phạm vi:** Dataset phiên bản *Current* ghi nhận các giao dịch phát sinh từ đầu năm 2025 đến hiện tại (các giao dịch trước 2025 được lưu trữ riêng tại bộ dữ liệu *Accounts Payable Archive*).
- **Giấy phép (License):** Áp dụng giấy phép **CC0 1.0 Universal (Public Domain Dedication)** — cho phép tự do khai thác, sao chép, phân tích và công bố phục vụ mục đích nghiên cứu, học tập mà không bị ràng buộc bản quyền thương mại.
- **Phương pháp thu thập & Hạn chế (Limitations / Biases):**
  - *Phương pháp:* Dữ liệu thứ cấp (secondary data) được trích xuất tự động từ hệ thống quản lý tài chính/ERP nội bộ của Phòng Tài chính thành phố Boulder.
  - *Hạn chế 1 (Thiên lệch địa lý cục bộ):* Chỉ phản ánh hoạt động chi tiêu của chính quyền đô thị Boulder, không đại diện cho cấu trúc chi tiêu của bang Colorado hay toàn nước Mỹ.
  - *Hạn chế 2 (Phạm vi chi phí):* Chỉ bao gồm các khoản thanh toán cho nhà thầu/nhà cung ứng bên ngoài (Accounts Payable), không bao gồm chi trả lương trực tiếp cho đội ngũ công chức nội bộ (chi phí Payroll được bảo mật riêng).
  - *Hạn chế 3 (Nghiệp vụ kế toán):* Xuất hiện các giá trị số tiền âm do các nghiệp vụ giảm trừ công nợ, hoàn trả hàng hóa hoặc chứng từ ghi có (Credit Memo).
- **Động lực chọn đề tài (Motivation):**
  - Chi tiêu công là bức tranh minh bạch nhất về cách chính quyền vận hành ngân sách từ tiền thuế của người dân. 
  - Khám phá tập dữ liệu này giúp nhóm phân tích cơ cấu ngân sách đô thị, đánh giá mức độ phụ thuộc vào các nhà thầu lớn, phát hiện các bất thường trong dòng tiền giải ngân, và xây dựng mô hình tự động hóa phân loại danh mục chi tiêu hỗ trợ công tác kiểm toán công.


# 2. Data Exploration (Khám phá dữ liệu)

## 2.1 Khởi tạo môi trường & Tải dữ liệu (Environment Setup & Data Loading)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình thẩm mỹ chuẩn học thuật cho biểu đồ và bảng hiển thị
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams.update({
    "font.size": 11,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.edgecolor": "#cccccc",
    "axes.linewidth": 0.8
})
pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 2)

# Thiết lập đường dẫn tương đối từ thư mục notebooks/ và nạp dữ liệu
DATA_PATH = os.path.join("..", "data", "raw", "Accounts_Payable_Current.csv")
df = pd.read_csv(DATA_PATH)

print(f"Kích thước tập dữ liệu: {df.shape[0]:,} dòng, {df.shape[1]} cột")
df.head()


## 2.2 Từ điển dữ liệu (Data Dictionary)

Bảng dữ liệu gồm **35,251 dòng** và **13 cột**, phản ánh từng dòng chi tiết (line-item) trong các giao dịch chi trả công nợ và mua sắm công của Thành phố Boulder.

| Tên cột | Kiểu dữ liệu hiện tại | Kiểu dữ liệu kỳ vọng | Phân loại biến | Mô tả nghiệp vụ | Ví dụ minh họa |
| :--- | :---: | :---: | :---: | :--- | :--- |
| `vendor_name` | `object` | `string` / `category` | Categorical (Nominal) | Tên nhà cung cấp / tổ chức / cá nhân nhận thanh toán | `NAPA Auto Parts` |
| `transaction_date` | `object` | `datetime64[ns]` | Temporal | Ngày giao dịch được ghi nhận vào sổ cái kế toán | `01/06/2025` |
| `transaction_amount` | `float64` | `float64` | Numerical (Continuous) | Giá trị thanh toán (USD). Có thể âm do nghiệp vụ hoàn tiền / credit memo | `24.90` |
| `fund` | `object` | `category` | Categorical (Nominal) | Mã định danh nguồn quỹ tài chính thanh toán | `7210FD` |
| `fund_desc` | `object` | `string` | Categorical (Nominal) | Tên mô tả nguồn quỹ | `Fleet Fund` |
| `program` | `object` | `category` | Categorical (Nominal) | Mã định danh chương trình / dự án công của thành phố | `1905PG` |
| `program_desc` | `object` | `string` | Categorical (Nominal) | Tên mô tả chương trình / dự án | `FF Fleet Repairs and Maintenance` |
| `cost_center` | `object` | `category` | Categorical (Nominal) | Mã trung tâm chi phí (đơn vị hạch toán ngân sách) | `1589CC` |
| `cost_center_desc` | `object` | `string` | Categorical (Nominal) | Tên mô tả trung tâm chi phí | `Fleet` |
| `department` | `object` | `category` | Categorical (Nominal) | Tên phòng ban / cơ quan chủ quản thực hiện chi tiêu | `Facilities and Fleet` |
| `spend_category` | `object` | `category` | Categorical (Nominal) | Mã định danh danh mục hàng hóa / dịch vụ mua sắm | `6102SC` |
| `spend_category_desc`| `object` | `string` | Categorical (Nominal) | Tên mô tả danh mục hàng hóa / dịch vụ mua sắm | `Flt R&M: Vehicle Parts` |
| `ObjectId` | `int64` | `int64` | Identifier | Mã định danh số nguyên tự tăng do cổng dữ liệu ArcGIS tự gán | `1` |

> **Quan sát về cấu trúc dữ liệu:**
> - **Cặp thuộc tính Mã - Diễn giải (Code-Description Pairs):** Dataset tồn tại 4 cặp cột phản ánh cùng một thực thể: (`fund`, `fund_desc`), (`program`, `program_desc`), (`cost_center`, `cost_center_desc`), (`spend_category`, `spend_category_desc`). Ở giai đoạn tiền xử lý, nhóm có thể loại bỏ các cột mã để giảm chiều dữ liệu.
> - **Chuyển đổi kiểu dữ liệu:** Cần chuyển `transaction_date` sang dạng thời gian chuẩn `datetime64[ns]` để phục vụ trích xuất thuộc tính chu kỳ (tháng, quý, ngày trong tuần).


## 2.3 Kiểm tra toàn vẹn & Phân tích trùng lặp (Data Integrity & Duplication Analysis)


In [ ]:
# Kiểm tra số lượng bản ghi trùng lặp
dup_all = df.duplicated().sum()

# Loại trừ mã định danh ObjectId để kiểm tra trùng lặp trên các thuộc tính nghiệp vụ
feature_cols = [c for c in df.columns if c != "ObjectId"]
dup_business = df.duplicated(subset=feature_cols).sum()

print(f"Số lượng bản ghi trùng lặp toàn bộ (bao gồm ObjectId): {dup_all}")
print(f"Số lượng bản ghi trùng lặp thuộc tính nghiệp vụ (không gồm ObjectId): {dup_business:,} ({dup_business/len(df)*100:.2f}%)")

# Hiển thị nhóm các dòng trùng lặp để kiểm tra bản chất giao dịch
(
    df[df.duplicated(subset=feature_cols, keep=False)]
    .sort_values(by=["vendor_name", "transaction_date", "transaction_amount"])
    .loc[:, ["vendor_name", "transaction_date", "transaction_amount", "department", "spend_category_desc", "ObjectId"]]
    .head(6)
)


### Nhận xét & Biện luận về các bản ghi trùng lặp (Duplicate Records)

- **Kết quả kiểm tra:**
  - Xét trên toàn bộ 13 cột (bao gồm `ObjectId`): **0 dòng trùng lặp**, do `ObjectId` là khóa tự tăng được hệ sinh thái ArcGIS tự động gán cho mỗi bản ghi khi xuất dữ liệu.
  - Xét trên 12 cột đặc trưng nghiệp vụ (loại trừ `ObjectId`): Phát hiện **1,570 dòng** (chiếm 4.45% tổng dữ liệu) có giá trị giống hệt nhau trên tất cả các trường.

- **Phân tích bản chất nghiệp vụ kế toán:**
  1. *Cấp độ chi tiết của dữ liệu (Granularity):* Dữ liệu trích xuất từ sổ cái kế toán phản ánh từng **dòng chi tiết của hóa đơn (Line-item level)** chứ không phải theo từng hóa đơn tổng (Invoice level).
  2. *Thực tế giao dịch:* Quan sát các mẫu bản ghi trùng lặp (ví dụ: cùng ngày `01/02/2025`, đơn vị `Facilities and Fleet` có nhiều dòng thanh toán riêng lẻ trị giá đúng `$9.09` hoặc `$2.50` cho `NAPA Auto Parts`), đây là các khoản mua sắm nhiều linh kiện/phụ tùng cùng loại trong cùng một phiếu mua hàng (Purchase Order).
  3. *Hạn chế của tập dữ liệu:* Do hệ thống dữ liệu mở đã ẩn đi các trường nhạy cảm như `Invoice_Number` hay `PO_Number`, các dòng chi tiết giống nhau về mặt số tiền và phân loại sẽ biểu hiện dưới dạng trùng lặp trên bề mặt dữ liệu.

- **Quyết định xử lý (Decision):**
  - **KHÔNG loại bỏ 1,570 bản ghi này.** Việc tùy tiện xóa bỏ sẽ làm hao hụt (underestimate) tổng giá trị chi tiêu thực tế của thành phố Boulder. Toàn bộ các dòng này sẽ được bảo toàn nguyên vẹn.


## 2.4 Kiểm định chất lượng dữ liệu & Phân tích giá trị khuyết (Data Health Check & Missing Data)


In [ ]:
# Bảng tổng kiểm định chất lượng các thuộc tính dữ liệu
health_check_df = pd.DataFrame({
    "Dtype": df.dtypes,
    "Non-Null Count": df.notnull().sum(),
    "Missing Count": df.isnull().sum(),
    "Missing Ratio (%)": (df.isnull().mean() * 100).round(2),
    "Unique Values": df.nunique(),
    "Sample Value": df.iloc[0]
})

# Kiểm tra xem 7 cột tổ chức có bị khuyết đồng thời trên cùng một nhóm dòng hay không
org_cols = ["fund", "fund_desc", "program", "program_desc", "cost_center", "cost_center_desc", "department"]
simultaneous_missing = df[org_cols].isnull().all(axis=1).sum()
print(f"Số bản ghi khuyết đồng thời toàn bộ 7 thông tin tổ chức/phòng ban: {simultaneous_missing}")

health_check_df


### Nhận xét về chất lượng dữ liệu khuyết thiếu (Missing Data Patterns)

- **Hiện trạng:**
  - Có đúng **7 cột** chứa dữ liệu khuyết: `fund`, `fund_desc`, `program`, `program_desc`, `cost_center`, `cost_center_desc`, và `department`.
  - Tất cả 7 cột này đều có chính xác **484 giá trị khuyết** (tương đương **1.37%** tổng dữ liệu).
  - Các thuộc tính quan trọng khác (`vendor_name`, `transaction_date`, `transaction_amount`, `spend_category_desc`) đạt độ toàn vẹn **100% (không có missing)**.

- **Bản chất cơ chế khuyết (Missing Mechanism):**
  - Toàn bộ 484 dòng này **bị khuyết cùng một lúc** trên cả 7 trường hạch toán đơn vị. Điều này chỉ ra cơ chế khuyết không ngẫu nhiên (MNAR/MAR): Đây là các khoản chi cấp toàn thành phố (City-wide expenses) hoặc các giao dịch vãng lai chưa được phân bổ về một trung tâm chi phí cụ thể trong hệ thống kế toán nội bộ.

- **Đề xuất xử lý cho giai đoạn Preprocessing (Plan):**
  - Do tỷ lệ khuyết rất nhỏ (1.37%) và các giao dịch này vẫn mang thông tin hợp lệ về số tiền (`transaction_amount`) và nhà cung ứng (`vendor_name`), nhóm **sẽ không xóa bỏ các dòng này**.
  - Trong bước tiền xử lý, nhóm sẽ điền khuyết (impute) bằng nhãn đại diện `"Unassigned"` hoặc `"Citywide"` để không làm mất mát thông tin dòng tiền công.


## 2.5 Khảo sát thuộc tính định lượng (Numerical Exploration: `transaction_amount`)


In [ ]:
amt = df["transaction_amount"]

# 1. Thống kê 5 số và các chỉ số hình dạng phân phối
desc = amt.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
skewness = amt.skew()

print("--- THỐNG KÊ MÔ TẢ TRANSACTION AMOUNT ---")
print(desc.apply(lambda x: f"${x:,.2f}"))
print(f"Độ lệch (Skewness): {skewness:.2f} (Lệch phải cực mạnh)")

# 2. Khảo sát các giao dịch âm và bằng 0
neg_df = df[df["transaction_amount"] < 0]
zero_df = df[df["transaction_amount"] == 0]

print(f"\nSố giao dịch âm: {len(neg_df)} ({len(neg_df)/len(df)*100:.2f}%) | Tổng tiền hoàn: ${neg_df['transaction_amount'].sum():,.2f}")
print(f"Số giao dịch bằng 0: {len(zero_df)} ({len(zero_df)/len(df)*100:.2f}%)")


In [ ]:
# Lọc các giao dịch dương để vẽ phân phối trên thang logarit
positive_amounts = df[df["transaction_amount"] > 0]["transaction_amount"]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Histogram với KDE trên thang Log10
sns.histplot(positive_amounts, log_scale=True, kde=True, ax=axes[0], color="#2b5c8f", bins=40)
axes[0].set_title("Phân phối giá trị giao dịch (Thang Log10)")
axes[0].set_xlabel("Số tiền (USD - Log Scale)")
axes[0].set_ylabel("Tần suất (Số giao dịch)")

# 2. Boxplot phát hiện Outlier
sns.boxplot(x=positive_amounts, ax=axes[1], color="#e27c7c", log_scale=True)
axes[1].set_title("Biểu đồ Boxplot nhận diện ngoại lai (Outliers)")
axes[1].set_xlabel("Số tiền (USD - Log Scale)")

sns.despine()
plt.tight_layout()
plt.show()


### Nhận xét về phân phối giá trị giao dịch (`transaction_amount`)

- **Đặc trưng phân phối (Distribution Shape):**
  - **Lệch phải cực đoan (Extreme Right-skewed):** Giá trị trung bình (Mean = **$9,185.96**) gấp hơn **24 lần** giá trị trung vị (Median = **$371.51**). Phần lớn giao dịch công nằm trong mức chi tiêu nhỏ và vừa (50% giao dịch nằm trong khoảng $81.84 đến $1,820.51).
  - Phân phối trên thang Log10 cho thấy dạng chuẩn gần đối xứng với đỉnh tập trung quanh mốc **$300 – $500**.

- **Giá trị ngoại lai (Outliers):**
  - Giá trị lớn nhất ghi nhận lên tới **$11,000,000.00** (11 triệu USD).
  - *Đánh giá tính hợp lý:* Trong nghiệp vụ chi tiêu công của chính quyền thành phố, các khoản chi hàng triệu USD thường là các hợp đồng xây dựng hạ tầng lớn, mua sắm máy móc thiết bị công nặng hoặc chuyển vốn ngân sách. Do đó, đây là **Outliers thực tế (Genuine Outliers)** chứ không phải lỗi nhập liệu. Nhóm sẽ bảo toàn các giá trị này thay vì cắt gọt thô bạo.

- **Giá trị âm (Negative Values):**
  - Có **184 giao dịch âm** (chiếm 0.52%), giá trị nhỏ nhất là **-$3,383.32**, tổng tiền hoàn là **-$48,031.88**.
  - *Bản chất kế toán:* Đây là các nghiệp vụ kế toán công điển hình: Giảm trừ công nợ (Credit memo), hoàn trả hàng hóa hoặc điều chỉnh sai sót chứng từ của các kỳ trước. 
  - *Đề xuất tiền xử lý:* Khi phân tích quy mô mua sắm gộp (Gross Spending), cần xử lý cẩn trọng hoặc tách riêng phần chi phí thuần (Net Spending = Tổng dương - Tổng hoàn).


## 2.6 Khảo sát thuộc tính phân loại & Mối quan hệ chi tiêu (Categorical & Spending Relationships)


In [ ]:
# Tổng hợp dữ liệu theo phòng ban: Số lượng giao dịch vs Tổng ngân sách chi tiêu
dept_summary = (
    df.assign(department=df["department"].fillna("Unassigned"))
    .groupby("department")
    .agg(
        Transaction_Count=("transaction_amount", "count"),
        Total_Spend_USD=("transaction_amount", "sum"),
        Mean_Spend_USD=("transaction_amount", "mean"),
        Median_Spend_USD=("transaction_amount", "median")
    )
    .sort_values(by="Total_Spend_USD", ascending=False)
)

# Trực quan hóa so sánh kép (Dual-panel Barplot)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top10_by_spend = dept_summary.head(10)
top10_by_count = dept_summary.sort_values(by="Transaction_Count", ascending=False).head(10)

# Panel 1: Top 10 phòng ban có Tổng chi tiêu cao nhất (Triệu USD)
sns.barplot(
    x=top10_by_spend["Total_Spend_USD"] / 1e6,
    y=top10_by_spend.index,
    ax=axes[0],
    palette="Blues_r"
)
axes[0].set_title("Top 10 Phòng ban có Tổng chi tiêu cao nhất (Triệu USD)")
axes[0].set_xlabel("Tổng chi tiêu (Triệu USD)")
axes[0].set_ylabel("")

# Panel 2: Top 10 phòng ban có Tần suất giao dịch nhiều nhất
sns.barplot(
    x=top10_by_count["Transaction_Count"],
    y=top10_by_count.index,
    ax=axes[1],
    palette="Greens_r"
)
axes[1].set_title("Top 10 Phòng ban có Tần suất giao dịch nhiều nhất")
axes[1].set_xlabel("Số lượng giao dịch (bản ghi)")
axes[1].set_ylabel("")

sns.despine()
plt.tight_layout()
plt.show()


### Nhận xét mối quan hệ giữa Phòng ban và Quy mô Chi tiêu

- **Sự phân hóa giữa Tần suất (Count) và Giá trị (Value):**
  - **Facilities and Fleet** dẫn đầu tuyệt đối về *số lượng giao dịch* (8,580 bản ghi, chiếm gần 25% toàn bộ dữ liệu), nhưng chủ yếu là các khoản chi lẻ định kỳ cho bảo trì phụ tùng, xăng xe (giá trị trung vị chỉ khoảng $80 - $150).
  - Ngược lại, **Utilities (Dịch vụ Công ích)** mới là cơ quan ngốn *tổng ngân sách lớn nhất* (vượt hơn 82 triệu USD), do sở hữu các khoản chi hợp đồng hạ tầng cấp thoát nước và năng lượng quy mô lớn.
  - Tương tự, **Housing and Human Services** dù chỉ có 1,630 giao dịch nhưng nằm trong Top 4 tổng chi tiêu (hơn 33 triệu USD), cho thấy các khoản trợ cấp và dự án an sinh có đơn giá trên mỗi lần giải ngân rất cao.


## 2.7 Tóm tắt Khám phá & Kế hoạch Tiền xử lý (Summary & Preprocessing Plan)

### 1. Các phát hiện cốt lõi (Key Observations)
1. **Quy mô & Phân bố:** Tập dữ liệu gồm **35,251 bản ghi** và **13 cột**, phản ánh chi tiêu mua sắm công của TP Boulder từ năm 2025. Phân phối chi tiêu bị **lệch phải cực đoan** (Mean = $9,185 vs Median = $371), với các khoản chi hạ tầng lớn lên đến 11 triệu USD.
2. **Cấu trúc cặp mã - mô tả:** Tồn tại 4 cặp thuộc tính Mã $\leftrightarrow$ Tên mô tả tương đương nhau (`fund`, `program`, `cost_center`, `spend_category`).
3. **Mô hình chi tiêu hai tầng (Two-tier spending):** Tồn tại sự phân hóa rõ rệt giữa các phòng ban chi tiêu vận hành thường nhật (nhiều giao dịch nhỏ như `Facilities and Fleet`) và các cơ quan dự án hạ tầng/an sinh (ít giao dịch nhưng giá trị khổng lồ như `Utilities`, `Housing`).

### 2. Các vấn đề chất lượng dữ liệu (Data Quality Issues)
- **Bản ghi trùng lặp (1,570 dòng):** Phát sinh do dữ liệu chi tiết hóa đơn (line-item level) không có trường khóa hóa đơn.
- **Dữ liệu khuyết thiếu (484 dòng - 1.37%):** Toàn bộ 7 trường tổ chức hạch toán bị khuyết đồng thời, là các khoản chi cấp toàn thành phố chưa gắn trung tâm chi phí cụ thể.
- **Giá trị âm (184 dòng):** Do nghiệp vụ hoàn trả/credit memo trong kế toán công.
- **Lỗi định dạng ngày tháng:** Cột `transaction_date` đang ở dạng chuỗi ký tự (`object`) và chứa một số giá trị năm dị biệt do lỗi gõ phím lịch sử (ví dụ: năm 0001, 1925, 3035).

### 3. Kế hoạch tiền xử lý cho giai đoạn tiếp theo (Preprocessing Plan)
1. **Làm sạch chuỗi thời gian:** Ép kiểu `transaction_date` sang `datetime64[ns]`, lọc bỏ hoặc hiệu chỉnh các bản ghi có năm bất hợp lý ngoài phạm vi dự án (giữ lại các giao dịch 2024–2026). Trích xuất các biến thời gian: `month`, `quarter`, `day_of_week`.
2. **Điền khuyết (Imputation):** Gán nhãn `"Unassigned"` / `"Citywide"` cho 484 dòng khuyết thông tin phòng ban, tuyệt đối không drop dữ liệu.
3. **Loại bỏ thuộc tính thừa (Feature Selection):** Loại bỏ cột kỹ thuật `ObjectId` và các cột mã code trùng lặp để tinh gọn không gian đặc trưng.
4. **Phân tách phân tích:** Khi phân tích thống kê dòng tiền, xử lý riêng biệt giữa tổng chi gộp (Gross Spend) và chi phí hoàn trả (Credit Memos).
